# 🔮 Tarot Card Classifier — Training Notebook
EfficientNet-B0 fine-tuned on your tarot card photos.

**Before running:** Make sure Runtime > Change runtime type > **T4 GPU** is selected.

## Step 1: Install Dependencies

In [ ]:
!pip install -q torch torchvision tqdm
!pip install pillow-heif

## Step 2: Mount Google Drive

In [ ]:
from pillow_heif import register_heif_opener
register_heif_opener()  # allows PIL to open .heic files

from google.colab import drive
drive.mount('/content/drive')

## Step 3: Set Your Data Path

In [ ]:
import os

DATA_DIR = '/content/drive/MyDrive/Tarot-card'
SAVE_DIR = '/content/drive/MyDrive/fortune-teller/checkpoints'

os.makedirs(SAVE_DIR, exist_ok=True)

# Check card folders are visible
folders = sorted(os.listdir(DATA_DIR))
print(f'Total cards: {len(folders)}')
for f in folders:
    count = len(os.listdir(os.path.join(DATA_DIR, f)))
    print(f'  {f}: {count} images')

## Step 4: Model Definition

In [ ]:
import torch
import torch.nn as nn          # building blocks: Linear, Dropout, ReLU
from torchvision import models  # pretrained models: efficientnet_b0

class TarotClassifier(nn.Module):
    def __init__(self, num_classes=78, dropout=0.3):
        super().__init__()  # let PyTorch set itself up first

        # Load pretrained EfficientNet-B0 (trained on ImageNet)
        self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

        # Get input size of the original Linear layer (= 1280)
        in_features = self.backbone.classifier[1].in_features

        # Replace the original head with our custom head for 78 tarot classes
        self.backbone.classifier[1] = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(512, num_classes)  # 78 outputs = 78 cards
        )

    def forward(self, x):
        # x = batch of images → scores for each card
        return self.backbone(x)

    def freeze_backbone(self):
        # Phase 1: freeze feature layers + original dropout, only train our head
        for param in self.backbone.features.parameters():
            param.requires_grad = False
        for param in self.backbone.classifier[0].parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        # Phase 2: unfreeze everything for fine-tuning
        for param in self.backbone.parameters():
            param.requires_grad = True

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## Step 5: Dataset & DataLoaders

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split  # load and batch data
from torchvision import transforms  # resize, crop, normalize
from pathlib import Path

# ImageNet normalization — must match what EfficientNet was trained on
# normalized = (pixel - MEAN) / STD
MEAN = [0.485, 0.456, 0.406]  # R, G, B channel means
STD  = [0.229, 0.224, 0.225]  # R, G, B channel stds

# Training: augmentation + normalization
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),           # randomly cut 224x224 from 256x256
    transforms.RandomHorizontalFlip(),    # randomly mirror left-right
    transforms.RandomRotation(15),        # randomly rotate ±15 degrees
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),                # convert image to numbers 0.0-1.0
    transforms.Normalize(MEAN, STD),      # scale to ImageNet range
])

# Validation/Test: no augmentation — measure real accuracy
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class TarotDataset(Dataset):
    def __init__(self, root, transform):
        self.root = Path(root)       # path to data folder
        self.transform = transform   # train or val transform

        # Scan and sort all subfolders (each folder = one card class)
        class_dirs = sorted([d for d in self.root.iterdir() if d.is_dir()])
        self.classes = [d.name for d in class_dirs]               # ['00_the_fool', '01_the_magician', ...]
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}  # {'00_the_fool': 0, ...}

        # Build flat list of (image_path, label) for every image
        self.samples = []
        for d in class_dirs:
            label = self.class_to_idx[d.name]  # folder name → number
            for img_path in d.iterdir():
                if img_path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.heic'}:
                    self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)  # total number of images

    def __getitem__(self, idx):
        path, label = self.samples[idx]          # get path and label by index
        image = Image.open(path).convert('RGB')  # open image file
        return self.transform(image), label       # apply transforms → (tensor, label)

# Build dataset and split into train / val / test
full_ds = TarotDataset(DATA_DIR, train_transform)
n       = len(full_ds)
n_val   = int(n * 0.15)
n_test  = int(n * 0.10)
n_train = n - n_val - n_test

train_ds, val_ds, test_ds = random_split(full_ds, [n_train, n_val, n_test])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2)

print(f'Train: {n_train} | Val: {n_val} | Test: {n_test}')
print(f'Classes: {len(full_ds.classes)}')

## Step 6: Training Loop

In [ ]:
import json
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.notebook import tqdm

def accuracy(outputs, labels):
    return (outputs.argmax(dim=1) == labels).float().mean().item()

def run_epoch(model, loader, criterion, optimizer, train):
    model.train() if train else model.eval()
    total_loss, total_acc = 0.0, 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for images, labels in tqdm(loader, leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            total_acc  += accuracy(outputs, labels)
    n = len(loader)
    return total_loss / n, total_acc / n

# Save class index mapping (needed for prediction later)
with open(f'{SAVE_DIR}/classes.json', 'w') as f:
    json.dump(full_ds.classes, f, indent=2)

model = TarotClassifier(num_classes=len(full_ds.classes)).to(DEVICE)
model.freeze_backbone()
criterion = nn.CrossEntropyLoss()
best_val_acc = 0.0

# Phase 1: Train head only (backbone frozen)
print('=== Phase 1: Train head only ===')
optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
scheduler = CosineAnnealingLR(optimizer, T_max=10)

for epoch in range(1, 11):
    tl, ta = run_epoch(model, train_loader, criterion, optimizer, train=True)
    vl, va = run_epoch(model, val_loader,   criterion, optimizer, train=False)
    scheduler.step()
    print(f'[P1 {epoch:02d}/10] loss={tl:.4f} acc={ta:.3f} | val_loss={vl:.4f} val_acc={va:.3f}')
    if va > best_val_acc:
        best_val_acc = va
        torch.save(model.state_dict(), f'{SAVE_DIR}/best_model.pt')
        print(f'  Saved best model (val_acc={va:.3f})')

# Phase 2: Fine-tune full model (backbone unfrozen)
print('\n=== Phase 2: Fine-tune full model ===')
model.unfreeze_backbone()
optimizer = Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=20)

for epoch in range(1, 21):
    tl, ta = run_epoch(model, train_loader, criterion, optimizer, train=True)
    vl, va = run_epoch(model, val_loader,   criterion, optimizer, train=False)
    scheduler.step()
    print(f'[P2 {epoch:02d}/20] loss={tl:.4f} acc={ta:.3f} | val_loss={vl:.4f} val_acc={va:.3f}')
    if va > best_val_acc:
        best_val_acc = va
        torch.save(model.state_dict(), f'{SAVE_DIR}/best_model.pt')
        print(f'  Saved best model (val_acc={va:.3f})')

## Step 7: Test Evaluation

In [ ]:
model.load_state_dict(torch.load(f'{SAVE_DIR}/best_model.pt'))
test_loss, test_acc = run_epoch(model, test_loader, criterion, None, train=False)
print(f'Test Loss: {test_loss:.4f}  |  Test Accuracy: {test_acc:.3f}')

## Step 8: Test 5 Random Cards

In [ ]:
import matplotlib.pyplot as plt
import random

model.load_state_dict(torch.load(f'{SAVE_DIR}/best_model.pt'))
model.eval()

# Pick 5 random samples from test set
test_indices = random.sample(range(len(test_ds)), 5)

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.patch.set_facecolor('#141414')

for ax, idx in zip(axes, test_indices):
    img_path, true_label = full_ds.samples[test_ds.indices[idx]]

    # Open and predict
    image = Image.open(img_path).convert('RGB')
    tensor = val_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
    top_probs, top_idxs = probs.topk(3)

    pred_name  = full_ds.classes[top_idxs[0]]
    confidence = top_probs[0].item() * 100
    true_name  = full_ds.classes[true_label]
    correct    = (top_idxs[0].item() == true_label)

    # Show image
    ax.imshow(image)
    ax.axis('off')
    ax.set_facecolor('#141414')

    # Green title = correct, Red = wrong
    color = '#27AE60' if correct else '#E74C3C'
    ax.set_title(
        f"Pred: {pred_name}\n{confidence:.1f}%\nTrue: {true_name}",
        fontsize=9, color=color, pad=6
    )

    # Print top-3 in console
    print(f"\n--- Sample {idx} ---")
    print(f"  True label : {true_name}")
    for p, i in zip(top_probs, top_idxs):
        mark = " ✓" if i.item() == true_label else ""
        print(f"  {full_ds.classes[i]:35s} {p.item()*100:.1f}%{mark}")

plt.suptitle(
    "5 Random Test Cards   |   Green = Correct     Red = Wrong",
    fontsize=13, color='white', y=1.02
)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/test_5_cards.png', bbox_inches='tight',
            facecolor='#141414', dpi=150)
plt.show()
print("\nSaved → checkpoints/test_5_cards.png")